# Problem 4: The central dogma and open reading frames
-----------

Create a python class that emulates the central dogma of molecular biology: *transcription from DNA to RNA and translation from RNA to protein*.  The class should take a sequence and description as input and conduct the transcription and translation.  

Using your FASTA parser from Problem 2, imported from your package, write a program that reads in a file in FASTA format, passes the description and sequence to your class, and determines the resulting protein sequence.  

Print out **all the open reading frames** that are potential coding sequences (i.e. have a start/stop codon with amino acids in between). If the sequence does not meet the criteria for a coding sequence, do not print it out.

## Rules
- Search all six reading frames: frames 1, 2 and 3 begin at bases 1, 2 and 3 of the sequence; frames -1, -2 and -3 read the reverse complement, beginning at the last, second-to-last and third-to-last base of the sequence
- An ORF begins at an `ATG` start codon and ends at the first stop codon (`TAA`, `TAG` or `TGA`) that follows it in the same frame
- Every `ATG` begins an ORF, including an `ATG` inside another ORF. Nested ORFs share a stop codon, and each is listed
- A start codon with no stop codon after it in the same frame is not an ORF
- There is no minimum length. Later modules add rules for deciding which ORFs are likely genes
- Positions are 1-based and inclusive, and always counted on the given (forward) sequence. `start` is the first base of the `ATG`; `stop` is the last base of the stop codon. In frames -1 to -3, `start` is therefore larger than `stop`
- `length` is the number of amino acids, not counting the stop codon. The protein sequence does not include a stop symbol

## Output
The output should follow the following format:
```
* frame_position [-3,-2,-1,1,2,3] | starting_base_position | ending_base_position | protein_sequence_length | protein sequence
```
Example output:
```
frame | start | stop | length | sequence 
1 | 1 | 24 | 7 | MKTAYIA
-2 | 34 | 5 | 9 | MSLVQEKGR
```

Use the following sequence file (`data/dogma.fasta`) as input:

```
>gi|XXXXXXXXX
GTACGACGGAGTGTTATAAGATGGGAAATCGGATACCAGATGAAATTGTGGATCAGGTGCAAAAGTCGGC
AGATATCGTTGAAGTCATAGGTGATTATGTTCAATTAAAGAAGCAAGGCCGAAACTACTTTGGACTCTGT
CCTTTTCATGGAGAAAGCACACCTTCGTTTTCCGTATCGCCCGACAAACAGATTTTTCATTGCTTTGGCT
GCGGAGCGGGCGGCAATGTTTTCTCTTTTTTAAGGCAGATGGAAGGCTATTCTTTTGCCGAGTCGGTTTC
TCACCTTGCTGACAAATACCAAATTGATTTTCCAGATGATATAACAGTCCATTCCGGAGCCCGGCCAGAG
TCTTCTGGAGAACAAAAAATGGCTGAGGCACATGAGCTCCTGAAGAAATTTTACCATCATTTGTTAATAA
ATACAAAAGAAGGTCAAGAGGCACTGGATTATCTGCTTTCTAGGGGCTTTACGAAAGAGCTGATTAATGA
ATTTCAGATTGGCTATGCTCTTGATTCTTGGGACTTTATCACGAAATTCCTTGTAAAGAGGGGATTTAGT
GAGGCGCAAATGGAAAAAGCGGGTCTCCTGATCAGACGCGAAGACGGAAGCGGATATTTCGACCGCTTCA
GAAACCGTGTCATGTTTCCGATCCATGATCATCACGGGGCTGTTGTTGCTTTCTCAGGCAGGGCTCTTGG
```


In [9]:
import sys

from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from utilities.fatsa_parser import read_fasta

class ProteinSyn:

    def __init__(self, description, sequence):
        self.description = description
        self.sequence = sequence.upper()

    # DNA -> RNA conversion- transcription
    def transcription(self, sequence):
        return sequence.replace("T", "U")

    # Generate reverse complement of DNA for generating the template strand and getting the reading frames
    def reverse_complement(self, sequence):

        watson_pairs = {
            "A": "T",
            "T": "A",
            "C": "G",
            "G": "C"
        }

        complement = ""

        for base in sequence:
            complement += watson_pairs[base]

        return complement[::-1]


    # RNA -> Protein translation
    def translation(self, rna_sequence):

        codon_table = {
            "UUU": "F", "UUC": "F",
            "UUA": "L", "UUG": "L",
            "UCU": "S", "UCC": "S", "UCA": "S", "UCG": "S",
            "UAU": "Y", "UAC": "Y",
            "UAA": "*", "UAG": "*",
            "UGU": "C", "UGC": "C",
            "UGA": "*", "UGG": "W",

            "CUU": "L", "CUC": "L", "CUA": "L", "CUG": "L",
            "CCU": "P", "CCC": "P", "CCA": "P", "CCG": "P",
            "CAU": "H", "CAC": "H",
            "CAA": "Q", "CAG": "Q",
            "CGU": "R", "CGC": "R", "CGA": "R", "CGG": "R",

            "AUU": "I", "AUC": "I", "AUA": "I",
            "AUG": "M",
            "ACU": "T", "ACC": "T", "ACA": "T", "ACG": "T",
            "AAU": "N", "AAC": "N",
            "AAA": "K", "AAG": "K",
            "AGU": "S", "AGC": "S",
            "AGA": "R", "AGG": "R",

            "GUU": "V", "GUC": "V", "GUA": "V", "GUG": "V",
            "GCU": "A", "GCC": "A", "GCA": "A", "GCG": "A",
            "GAU": "D", "GAC": "D",
            "GAA": "E", "GAG": "E",
            "GGU": "G", "GGC": "G", "GGA": "G", "GGG": "G"
        }

        protein = ""

        for i in range(0, len(rna_sequence) - 2, 3):

            # getting non overlapping groups of 3 letters
            codon = rna_sequence[i:i + 3]

            # not including stop codon in protein
            if codon in ("UAA", "UAG", "UGA"):
                break

            protein += codon_table[codon]

        return protein


    # Finding open reading frames(ORF) in one DNA strand
    def find_orfs_in_strand(self, dna_sequence, frame, reverse=False):

        stop_codons = {"TAA", "TAG", "TGA"}
        orfs = []

        # 3 frames with 3 offsets
        offset = abs(frame) - 1

        for i in range(offset, len(dna_sequence) - 2, 3):
            codon = dna_sequence[i:i + 3]

            # Every ATG begins a potential ORF
            if codon == "ATG":

                # Search for first stop codon in same frame
                for j in range(i + 3, len(dna_sequence) - 2, 3):
                    stop = dna_sequence[j:j + 3]
                    if stop in stop_codons:

                        # DNA from ATG up to the stop codon, excluding the stop codon
                        coding_dna = dna_sequence[i:j]

                        # DNA -> RNA
                        rna = self.transcription(coding_dna)

                        # RNA -> protein
                        protein = self.translation(rna)

                        if not reverse:
                            # indexing with 1-based coordinates
                            start_position = i + 1

                            # j is first base of stop codon, and +3 gives its final base in 1-based indexing.
                            stop_position = j + 3

                        else:
                            # Convert reverse-complement positions
                            # back to coordinates on original sequence
                            n = len(self.sequence)

                            start_position = n - i
                            stop_position = n - j - 2

                        orfs.append({
                            "frame": frame,
                            "start": start_position,
                            "stop": stop_position,
                            "length": len(protein),
                            "protein": protein
                        })

                        # ORF ends at first break
                        break

        return orfs


    def find_all_orfs(self):

        all_orfs = []

        # in forward strands
        for frame in [1, 2, 3]:

            orfs = self.find_orfs_in_strand(
                self.sequence,
                frame,
                reverse=False
            )

            all_orfs.extend(orfs)

        # getting the template strand
        reverse_seq = self.reverse_complement(self.sequence)

        # going from the back
        for frame in [-1, -2, -3]:

            orfs = self.find_orfs_in_strand(
                reverse_seq,
                frame,
                reverse=True
            )

            all_orfs.extend(orfs)

        return all_orfs

In [10]:
file_path = "../data/dogma.fasta"

print("frame | start | stop | length | sequence")

for description, sequence in read_fasta(file_path):
    protein_syn = ProteinSyn(description, sequence)
    orfs = protein_syn.find_all_orfs()
    for orf in orfs:

        print(
            f"{orf['frame']} | "
            f"{orf['start']} | "
            f"{orf['stop']} | "
            f"{orf['length']} | "
            f"{orf['protein']}"
        )

frame | start | stop | length | sequence
1 | 40 | 90 | 16 | MKLWIRCKSRQISLKS
1 | 97 | 108 | 3 | MFN
1 | 148 | 243 | 31 | MEKAHLRFPYRPTNRFFIALAAERAAMFSLF
1 | 226 | 243 | 5 | MFSLF
1 | 316 | 324 | 2 | MI
1 | 382 | 393 | 3 | MSS
1 | 487 | 546 | 19 | MNFRLAMLLILGTLSRNSL
1 | 505 | 546 | 13 | MLLILGTLSRNSL
-1 | 643 | 503 | 46 | MTRFLKRSKYPLPSSRLIRRPAFSICASLNPLFTRNFVIKSQESRA
-1 | 88 | 53 | 11 | MTSTISADFCT
-3 | 662 | 414 | 82 | MIMDRKHDTVSEAVEISASVFASDQETRFFHLRLTKSPLYKEFRDKVPRIKSIANLKFINQLFRKAPRKQIIQCLLTFFCIY
-3 | 656 | 414 | 80 | MDRKHDTVSEAVEISASVFASDQETRFFHLRLTKSPLYKEFRDKVPRIKSIANLKFINQLFRKAPRKQIIQCLLTFFCIY
-3 | 410 | 240 | 56 | MMVKFLQELMCLSHFLFSRRLWPGSGMDCYIIWKINLVFVSKVRNRLGKRIAFHLP
-3 | 407 | 240 | 55 | MVKFLQELMCLSHFLFSRRLWPGSGMDCYIIWKINLVFVSKVRNRLGKRIAFHLP
-3 | 383 | 240 | 47 | MCLSHFLFSRRLWPGSGMDCYIIWKINLVFVSKVRNRLGKRIAFHLP
-3 | 332 | 240 | 30 | MDCYIIWKINLVFVSKVRNRLGKRIAFHLP
-3 | 200 | 105 | 31 | MKNLFVGRYGKRRCAFSMKRTESKVVSALLL
-3 | 149 | 105 | 14 | MKRTESKVVSALLL


## Follow-up: is an ORF a gene?

No code for this part. Answer in a few sentences.

Three of the 64 possible codons are stop codons. If you picked codons at random, roughly how many would you expect to read before reaching a stop? Using that, would you expect a random DNA sequence to contain ORFs? What does your answer suggest about whether each ORF your program found is a real gene?


Answer:

Three of the 64 possible codons are stop codons, so the probability of encountering a stop codon at any codon is

$$
p = \frac{3}{64}.
$$

The number of codons read before encountering a stop can be modeled using a geometric distribution. The expected number of codons until a stop codon is encountered is

$$
E[X] = \frac{1}{p}
= \frac{1}{3/64}
= \frac{64}{3}
\approx 21.3.
$$

Therefore, we would expect to encounter a stop codon roughly every 21 codons in a random sequence.

Because both start codons (ATG) and stop codons (TAA, TAG, and TGA) can occur by chance, we can expect sufficiently long random DNA sequences to contain ORFs. So, simply finding an ORF does not mean that it represents a real gene. Additional biological evidence would be required to determine whether an ORF is actually a protein-coding gene.